## LangChain 实现RAG

LangChain提供了一系列工具、套件和接口，可以简化创建由LLMs和聊天模型提供支持的应用程序的
过程。LangChain包含6个组件：
- **模型（Models）**，包含各大语言模型的LangChain接口和调用细节，以及输出解析机制。
- **提示模板（Prompts）**，使提示工程流线化，进一步激发大语言模型的潜力。
- **数据检索（Indexes）**，构建并操作文档的方法，接受用户的查询并返回最相关的文档，轻松搭建本地知识库。
- **记忆（Memory）**，通过短时记忆和长时记忆，在对话过程中存储和检索数据，让ChatBot记住你是谁。
- **链（Chains）**，是LangChain中的核心机制，以特定方式封装各种功能，并通过一系列的组合，自动而灵活地完成常见用例。
- **代理（Agents）**，通过“代理”让大模型自主调用外部工具和内部工具，使强大的“智能化”自主Agent成为可能！

In [1]:
! pip install -r requirements.txt

  Using cached langchain-1.0.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_core-1.4.3-py3-none-any.whl.metadata (4.5 kB)
  Using cached langchain_openai-1.0.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached langchain_text_splitters-1.0.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached langchain_huggingface-1.0.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached langchain_chroma-1.0.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached torch-2.8.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (30 kB)
  Using cached torchvision-0.23.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.1 kB)
  Using cached langgraph-1.0.10-py3-none-any.whl.metadata (7.4 kB)
INFO: pip is looking at multiple versions of langchain-classic to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━

### 1. 基于 request库 实现的简单爬虫 -> 爬取百度百科的内容

In [2]:
import requests
from bs4 import BeautifulSoup
url = "https://baike.baidu.com/item/%E8%97%9C%E9%BA%A6/5843874"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")
soup

<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8"/><meta content="IE=edge" http-equiv="X-UA-Compatible"/><meta content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no" name="viewport"/><title>百度安全验证</title><style>body{margin:0;padding:0}</style></head><body></body><!--loadScript--><script>function ieOnEnd(script,cb){script.onreadystatechange=function(){if(this.readyState!=='complete'&&this.readyState!=='loaded'){return}this.onreadystatechange=null;cb(null,script)}}function stdOnEnd(script,cb){script.onload=function(){this.onerror=this.onload=null;cb(null,script)};script.onerror=function(e){this.onerror=this.onload=null;cb(new Error(`Failed to load ${this.src}`),script)}}function loadScript(src,opts,cb){var script=document.createElement('script');if(typeof opts==='function'){cb=opts;opts={}}opts=opts||{};cb=cb||function(){};script.type=opts.type||'text/javascript';script.charset=opts.charset||'utf8';script.async='async'in opts?!!opts.async:true;scri

In [3]:
# 创建列表存储提取结果
extracted_content = []

# 提取所有标题和段落
for element in soup.find_all('div', 'h2', 'h3'):
  # 处理一级标题
  if element.name == 'div' and 'level-1' in element.get('class', []):
    h2_tag = element.find('h2')
    if h2_tag:
      extracted_content.append(f"\n[标题] {h2_tag.get_text(strip=True)}")

  # 处理二级标题
  elif element.name == 'div' and 'level-2' in element.get('class', []):
    h3_tag = element.find('h3')
    if h3_tag:
      extracted_content.append(f"\n [小标题] {h3_tag.get_text(strip=True)}")

  # 处理正文段落
  elif element.name == 'div' and 'para_e7uVn' in element.get('class', []):
     # 提取所有文本内容
     text_spans = element.find_all('span', class_="text_jc0vX")
     if text_spans:
        paragraph = ''.join(span.get_text(strip=True) for span in text_spans)
        extracted_content.append(paragraph)

with open('./藜麦.txt', w, encoding='utf-8') as f:
  for item in extracted_content:
    f.wtire(item + '\n')


print("内容已成功提取到藜麦.txt文件")

SyntaxError: expected ':' (3461912539.py, line 7)

文本分割时主要需要考虑下面几点：
1. 如何分割
2. 块大小
3. 块之间重合长度

**Langchain提供了以下文本分割器:**

| 分割器 | 作用 |
|--------|------|
| `RecursiveCharacterTextSplitter` | 按 `\n\n`、`\n`、空格等递归切分，尽量保持段落/句子完整，**最常用首选** |
| `CharacterTextSplitter` | 按单个分隔符（默认 `\n\n`）切分，最基础 |
| `TokenTextSplitter` | 按 token 数切分，精确控制长度以匹配 LLM 上限 |
| `MarkdownHeaderTextSplitter` | 按标题层级切分，并将标题写入 metadata |

**核心参数：**

- `chunk_size`：每块最大长度
- `chunk_overlap`：相邻块重叠长度，保持上下文连贯

> 💡 通用场景直接用 `RecursiveCharacterTextSplitter` 即可。

In [1]:
# 上传文本
from langchain_community.document_loaders import TextLoader
loader = TextLoader("./藜麦.txt")
documents = loader.load()
documents

[Document(metadata={'source': './藜麦.txt'}, page_content='藜（读音lí）麦（ChenopodiumquinoaWilld.）是藜科藜属植物。植株形状类似藜（灰灰菜），成熟后上部花穗部分稍类似高粱穗，可呈红色、紫色或黄色。植株大小受环境及遗传因素影响较大，从0.3-3 m不等，茎部质地较硬，可分枝可不分。单叶互生，叶片呈鸭掌状，叶缘分为全缘型与锯齿缘型。藜麦花两型，花序呈伞状、穗状、圆锥状，藜麦种子较小，呈小圆药片状，直径1.5～2 mm，千粒重1.4～3 g。\n藜麦原产于南美洲安第斯山脉的哥伦比亚、厄瓜多尔、秘鲁等中高海拔山区，生长范围约为海平面到海拔4500 m左右的高原上，最适的高度为海拔3000-4000 m的高原或山地地区。藜麦适应性强，具有一定的耐旱、耐寒、耐盐性。\n与其他谷物相比，藜麦中脂肪含量较高；藜麦富含的维生素、多酚、类黄酮类、皂苷和植物甾醇类物质具有多种健康功效。藜麦具有高蛋白，其所含脂肪中不饱和脂肪酸占83%，藜麦也是一种低葡萄糖的食品，在糖脂代谢过程中发挥有利功效。\n（概述图参考来源：）\n姜再定希妹估驼藜麦(Chenopodium quinoa 姜拒悼Willd.)起源于南艰盛美洲提提喀喀湖欠射阀区，堡凝是苋科(Amaranthaceae)藜属(Chenopod永希ium)一年生作物。早在公元前5000年左右就被南美提提喀喀湖区土著居民筛选为主粮作物，被称为“粮食乊母”，养育了印加文明。巩背格20世纪70年代，藜麦被美国航空航天局选作太空主食，营养学家称其为“营养黄釐”、“超级谷物”。目前普遍认为藜麦是一种全营养食物。\n藜麦成熟后上部花穗部分稍类似高粱穗，可呈红色、紫色或黄色。植株大小受环境及遗传因素影响较大，从0.3-3 m不等，茎部质地较硬，可分枝可不分。单叶互生，叶片呈鸭掌状，叶缘分为全缘型与锯齿缘型。根系庞大但分布较浅，根上的须根多，吸水能力强。藜麦花两性，花序呈伞状、穗状、圆锥状，藜麦种子较小，呈小圆药片状，直径1.5～2 mm，千粒重1.4～3 g。\n藜麦\n大麦\n单叶互生，叶片呈鸭掌状，叶缘分为全缘型与锯齿缘型。\n叶鞘松散抱茎，无毛或基部者被柔毛；叶耳披针形；叶舌膜质。\n花序呈伞状、穗状、圆锥状。\n穗状花序稠密。\n该物种的原生地是厄瓜多尔到阿根廷西北部

这里选择使用 **RecursiverCharacterTextSplitter** 进行文本分块，防止存入向量数据库的文本快过长。

In [2]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=128,
    chunk_overlap=50,
    separators=["\n\n", "\n", "。", "!", "?", "//"]   # 优化分隔符
)

texts = text_splitter.split_documents(documents)
# 关键步骤: 提取所有文本内容
print(type(texts[0]))

<class 'langchain_core.documents.base.Document'>


使用 `langchain_text_splitters` 中的 `RecursiveCharacterTextSplitter` 进行分块。

先对要使用的文本分块器初始化，然后调用 `.split_documents` 进行切块，这样的好处是数据类型仍然是 `document` 并且包含元数据，这样可以很方便地存入向量数据库。

### **3. 嵌入模型 -> 将 chunks 存入向量数据库**

In [9]:
!pip install sentence-transformers
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 MB 16.2 MB/s eta 0:00:00
  Attempting uninstall: torch
    Found existing installation: torch 2.8.0
    Uninstalling torch-2.8.0:
      Successfully uninstalled torch-2.8.0


In [3]:
from huggingface_hub import snapshot_download

local_path = snapshot_download(
    repo_id="moka-ai/m3e-base",
    local_dir=".\models\m3e-base",
)

print("下载完成", local_path)

<>:5: SyntaxWarning: invalid escape sequence '\m'
<>:5: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_25986/530832078.py:5: SyntaxWarning: invalid escape sequence '\m'
  local_dir=".\models\m3e-base",


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

下载完成 /content/.\models\m3e-base


In [4]:
from langchain_community.vectorstores.pgembedding import EmbeddingStore
# 嵌入模型文本转换为数值表示
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import os

# 使用本地模型路径
model_path = r".\models\m3e-base"

# 确保模型的路径存在
if not os.path.exists(model_path):
  raise FileNotFoundError(f"模型路径不存在{model_path}, 请检查路径!")

# 创建 Embedding 模型的参数
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': True}

# 初始化 Embedding 模型
embeddings = HuggingFaceEmbeddings(
    model_name=model_path,
    model_kwargs=model_kwargs,
)

db = Chroma.from_documents(texts, embedding=embeddings)
# print(db.similarity_search("藜一般在几月播种?", k=3))

In [5]:
db

### **4. 构建一个能检索本地数据库的Agent**

RAG 智能体代码逻辑链条：

1. **读取 API Key** — 用 `userdata.get` 从 Colab 密钥中取出 `OPENAI_API_KEY`

2. **初始化 LLM** — `init_chat_model` 创建 gpt-4o 模型，`temperature=0`

3. **创建检索器** — `db.as_retriever()` 基于向量库生成检索器

4. **封装为工具** — 用 `@tool` 装饰 `search_kb`：接收 query，调用检索器取相关文档，拼接成文本返回

5. **创建 Agent** — `create_agent` 组合三要素：model（LLM）、tools（search_kb）、system_prompt（农业专家角色，要求优先检索再回答）

6. **调用 Agent** — `agent.invoke` 传入 `messages`（用户提问"藜麦怎么防治虫害?"）

7. **内部检索** — LLM 依据系统提示自主决定调用 `search_kb`，获取知识库上下文

8. **输出回答** — 从 `response['messages'][-1].content` 取出最终生成的答案

核心思路：**先准备底层组件（密钥、LLM、检索器）→ 把检索器封装成工具 → 组装成 Agent → 调用时由 LLM 自动决定检索并基于结果回答**。

In [14]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool

from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')

llm = init_chat_model(
    model="gpt-4o",
    model_provider="openai",
    api_key=api_key,
    temperature=0,
)

retriever = db.as_retriever()

@tool
def search_kb(query: str) -> str:
  """搜索知识库并返回相关内容"""
  docs = retriever.invoke(query)
  return "\n\n".join(doc.page_content for doc in docs)

agent = create_agent(
    model=llm,
    tools=[search_kb],
    system_prompt="你是农业领域的专家, 回答问题优先使用 search_kb 检索相关知识, 再基于检索结果回答。"
)

response = agent.invoke(
    {
        'messages': [{'role': 'user', 'content': '藜麦怎么防治虫害?'}]
    }
)

print(response)
print("===============================")
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='藜麦怎么防治虫害?', additional_kwargs={}, response_metadata={}, id='c973cdd4-8fb1-4f12-847f-bda88874dc11'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 87, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_00e0de3dc1', 'id': 'chatcmpl-Dp3WjFah5gQUv452HoKUz5Ys5SXrA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eaf7c-df68-7d11-b292-526d8775743c-0', tool_calls=[{'name': 'search_kb', 'args': {'query': '藜麦虫害防治'}, 'id': 'call_2r78MxE58KCeksfh0LVfUedx', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 87, 'output_tokens': 21, 'total_token

#### **agent.invoke() 的参数说明**

`agent.invoke()` 接收一个**字典**，代表 agent 的初始状态，其中 key 固定为 `messages`：

```python
response = agent.invoke(
    {'messages': [{'role': 'user', 'content': '藜麦怎么防治虫害?'}]}
)
```

**为什么 key 必须是 `messages`（复数）**：`create_agent` 底层基于 LangGraph 构建，内部维护一个状态对象，默认使用 `MessagesState` 结构，约定状态里必须有一个名为 `messages` 的字段、类型为消息列表。这个名字是固定约定，写成 `message` 或其他名字会因找不到字段而报错。

**为什么用消息列表形式**：每条消息是 `{'role': ..., 'content': ...}` 格式（`role` 可为 user / assistant / system / tool），这是主流 LLM API 的通用标准。用列表的核心原因是——agent 执行过程中会不断往 `messages` 里追加消息。一次 invoke 内部通常经历：用户提问（user）→ LLM 决定调用工具（assistant 的 tool call）→ 工具返回检索结果（tool）→ LLM 基于结果生成最终答案（assistant）。这些消息依次累加进列表。

**因此**取结果时用 `response['messages'][-1].content`，即列表最后一条消息的内容，就是 LLM 的最终回答。

可以把 `messages` 理解成一条会被 agent 不断写入的"对话流水账"：你传入开头，agent 跑完后整个过程都记录在里面，最后一条就是答案。